In [6]:
# ============================================================
# DNA SPLICE-JUNCTION SEQUENCE ANALYSIS AND CLASSIFICATION
# Dataset: UCI Molecular Biology Splice-junction Gene Sequences
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
# ------------------------------------------------------------
# 1. LOAD THE DATASET
# ------------------------------------------------------------
dataset_file = Path("data") / "splice.data"
if not dataset_file.is_file():
    raise FileNotFoundError(
        f"Dataset not found at: {dataset_file.resolve()}"
    )
print("Dataset found at:", dataset_file.resolve())
df = pd.read_csv(
    dataset_file,
    header=None,
    names=["class", "instance", "sequence"],
    sep=",",
    skipinitialspace=True,
    dtype=str
)
# Clean whitespace and standardise sequence case
for column in ["class", "instance", "sequence"]:
    df[column] = df[column].str.strip()
df["sequence"] = df["sequence"].str.upper()
print("\nOriginal dataset shape:", df.shape)
print("\nFirst five records:")
print(df.head().to_string(index=False))
# ------------------------------------------------------------
# 2. DATA QUALITY CHECKS
# ------------------------------------------------------------
print("\nMissing values:")
print(df.isnull().sum())
if df.isnull().any().any():
    raise ValueError(
        "Missing values detected. Inspect the dataset before training."
    )
# Remove exact duplicate rows
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("\nExact duplicate records removed:", duplicate_count)
print("Dataset shape after cleaning:", df.shape)
# Check sequence lengths
sequence_lengths = df["sequence"].str.len()
print("\nSequence length distribution:")
print(sequence_lengths.value_counts().sort_index())
if sequence_lengths.nunique() != 1:
    raise ValueError(
        "Sequences have different lengths. Preprocessing is required."
    )
sequence_length = int(sequence_lengths.iloc[0])
# Inspect nucleotide symbols
all_symbols = sorted(set("".join(df["sequence"])))
print("\nUnique sequence symbols:", all_symbols)
print("Sequence length:", sequence_length)
# Class distribution
print("\nClass distribution:")
print(df["class"].value_counts())
print("\nClass meanings:")
print("EI = Exon-to-intron junction (donor site)")
print("IE = Intron-to-exon junction (acceptor site)")
print("N  = Neither of the specified junction types")
# ------------------------------------------------------------
# 3. PREPARE FEATURES AND TARGET
# ------------------------------------------------------------
# X contains one feature for each nucleotide position.
# y contains the target class to predict.
X = pd.DataFrame(
    df["sequence"].map(list).tolist(),
    columns=[
        f"position_{i + 1}"
        for i in range(sequence_length)
    ],
    index=df.index
)
y = df["class"]
print("\nFeature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
# ------------------------------------------------------------
# 4. SPLIT INTO TRAINING AND TESTING SETS
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))
# Check exact sequence overlap between the two sets.
# Overlap can occur when identical sequences have different IDs.
train_sequences = set(df.loc[X_train.index, "sequence"])
test_sequences = set(df.loc[X_test.index, "sequence"])
overlap = train_sequences.intersection(test_sequences)
print("\nUnique training sequences:", len(train_sequences))
print("Unique testing sequences:", len(test_sequences))
print("Distinct sequences appearing in both sets:", len(overlap))
# ------------------------------------------------------------
# 5. BUILD THE LOGISTIC REGRESSION MODEL
# ------------------------------------------------------------
logistic_model = Pipeline([
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    ),
    (
        "classifier",
        LogisticRegression(max_iter=2000)
    )
])
# Train the model
logistic_model.fit(X_train, y_train)
print("\nLogistic Regression training completed.")
# Predict the test set
logistic_predictions = logistic_model.predict(X_test)
# Evaluate the model
logistic_accuracy = accuracy_score(
    y_test, logistic_predictions
)
logistic_f1 = f1_score(
    y_test,
    logistic_predictions,
    average="macro"
)
print("\n========== LOGISTIC REGRESSION RESULTS ==========")
print(f"Accuracy: {logistic_accuracy * 100:.2f}%")
print(f"Macro F1-score: {logistic_f1:.4f}")
print("\nClassification report:")
print(classification_report(
    y_test,
    logistic_predictions,
    labels=sorted(y.unique()),
    zero_division=0
))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(
    y_test,
    logistic_predictions,
    labels=sorted(y.unique())
))
# ------------------------------------------------------------
# 6. TRAIN AND EVALUATE RANDOM FOREST
# ------------------------------------------------------------
random_forest_model = Pipeline([
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])
random_forest_model.fit(X_train, y_train)
random_forest_predictions = random_forest_model.predict(X_test)
random_forest_accuracy = accuracy_score(
    y_test,
    random_forest_predictions
)
random_forest_f1 = f1_score(
    y_test,
    random_forest_predictions,
    average="macro"
)

print("\n========== RANDOM FOREST RESULTS ==========")
print(f"Accuracy: {random_forest_accuracy * 100:.2f}%")
print(f"Macro F1-score: {random_forest_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    random_forest_predictions,
    labels=sorted(y.unique()),
    zero_division=0
))
# ------------------------------------------------------------
# 7. COMPARE BOTH MODELS
# ------------------------------------------------------------
results_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy (%)": round(logistic_accuracy * 100, 2),
        "Macro F1-score": round(logistic_f1, 4)
    },
    {
        "Model": "Random Forest",
        "Accuracy (%)": round(random_forest_accuracy * 100, 2),
        "Macro F1-score": round(random_forest_f1, 4)
    }
])
print("\n========== MODEL COMPARISON ==========")
print(results_df.to_string(index=False))
# Select the model with the higher macro F1-score.
# Macro F1 gives equal weight to each class.
if random_forest_f1 > logistic_f1:
    best_model = random_forest_model
    best_model_name = "Random Forest"
else:
    best_model = logistic_model
    best_model_name = "Logistic Regression"
print("\nSelected model based on macro F1-score:", best_model_name)
# ------------------------------------------------------------
# 8. PREDICT A NEW DNA SEQUENCE
# ------------------------------------------------------------
class_meanings = {
    "EI": "Exon-to-intron junction (donor site)",
    "IE": "Intron-to-exon junction (acceptor site)",
    "N": "Neither of the specified splice-junction types"
}
def predict_splice_class(sequence):
    """
    Predict the splice-junction class of a DNA sequence.
    The input must have the same length as the training sequences.
    """
    sequence = sequence.strip().upper()
    if len(sequence) != sequence_length:
        print(
            f"Error: Sequence must contain exactly "
            f"{sequence_length} characters."
        )
        return None
    # Accept symbols present in the dataset.
    # The trained encoder handles categorical symbols.
    allowed_symbols = set(all_symbols)
    if not set(sequence).issubset(allowed_symbols):
        print(
            "Error: Sequence contains symbols not found "
            "in the training dataset."
        )
        return None
    sequence_features = pd.DataFrame(
        [list(sequence)],
        columns=X.columns
    )
    prediction = best_model.predict(sequence_features)[0]
    print("\n========== NEW SEQUENCE PREDICTION ==========")
    print("Input sequence:", sequence)
    print("Predicted class:", prediction)
    print(
        "Meaning:",
        class_meanings.get(prediction, "Unknown class")
    )
    return prediction


# Demonstrate prediction using one dataset sequence.
# This is a functionality test, NOT an independent test example.

sample_sequence = df["sequence"].iloc[0]

print("\nDemonstration using a dataset sequence:")
predict_splice_class(sample_sequence)

# ------------------------------------------------------------
# 9. PROJECT SUMMARY
# ------------------------------------------------------------

print("\n========== PROJECT SUMMARY ==========")
print("Original records: 3190")
print("Records after exact duplicate removal:", len(df))
print("Sequence length:", sequence_length)
print("Number of classes:", y.nunique())
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print(f"Logistic Regression accuracy: {logistic_accuracy * 100:.2f}%")
print(f"Random Forest accuracy: {random_forest_accuracy * 100:.2f}%")
print("Selected model:", best_model_name)
print("\nProject pipeline completed successfully!")



Dataset found at: C:\Users\Ajay Raj\Desktop\Excelra\DNA_Bioinformatics_project\Data\splice.data

Original dataset shape: (3190, 3)

First five records:
class           instance                                                     sequence
   EI   ATRINS-DONOR-521 CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCCTTCGAGCCAGTCTG
   EI   ATRINS-DONOR-905 AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCCCTCCGTGCCCCCGC
   EI   BABAPOE-DONOR-30 GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCGGGGGCACGGGGATG
   EI  BABAPOE-DONOR-867 GGGCTGCGTTGCTGGTCACATTCCTGGCAGGTATGGGGCGGGGCTTGCTCGGTTTTCCCC
   EI BABAPOE-DONOR-2817 GCTCAGCCCCCAGGTCACCCAGGAACTGACGTGAGTGTCCCCATCCCGGCCCTTGACCCT

Missing values:
class       0
instance    0
sequence    0
dtype: int64

Exact duplicate records removed: 12
Dataset shape after cleaning: (3178, 3)

Sequence length distribution:
sequence
60    3178
Name: count, dtype: int64

Unique sequence symbols: ['A', 'C', 'D', 'G', 'N', 'R', 'S', 'T']
Sequence length: 60

Class distribution